In [ ]:
from numba import cuda
import numpy as np

In [ ]:
# CUDA Kernel

@cuda.jit
def add_arrays(x, y, out):
    i = cuda.grid(1)
    if i<x.size:
        out[i] = x[i] + y[i]


In [ ]:
# Host Code

n = 1024
# create input arrays on CPU
a = np.arange(n, dtype=np.float32)
b = np.arange(n, dtype=np.float32)

# Allocate output array
c= np.zeros(n,dtype=np.float32)

# create GPU arrays
d_a = cuda.to_device(a)
d_b = cuda.to_device(b)
d_c = cuda.to_device(c)

# CUDA Configuration
threads_per_block = 256
# blocks_per_grid = math.ceil(n / threads_per_block)
blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

# Launche kernel
add_arrays[blocks_per_grid, threads_per_block](d_a, d_b, d_c)

# Copy result back to CPU
c = d_c.copy_to_host()

# Print result
print("First 10 results: ", c[:10])

First 10 results:  [ 0.  2.  4.  6.  8. 10. 12. 14. 16. 18.]


In [ ]:
cuda.gpus

In [ ]:
import numpy as np
from numba import cuda

# CUDA Kernel for 2D array addition
@cuda.jit
def add_arrays_2d(x, y, out):
    idx, idy = cuda.grid(2)
    if idx < x.shape[0] and idy < x.shape[1]:
        out[idx, idy] = x[idx, idy] + y[idx, idy]

# Host Code for 2D arrays
rows = 16
cols = 16
n_2d = rows * cols


a_2d = np.arange(n_2d, dtype=np.float32).reshape(rows, cols)
b_2d = np.arange(n_2d, dtype=np.float32).reshape(rows, cols)


c_2d = np.zeros_like(a_2d, dtype=np.float32)


d_a_2d = cuda.to_device(a_2d)
d_b_2d = cuda.to_device(b_2d)
d_c_2d = cuda.to_device(c_2d)


threads_per_block_2d = (16, 16)


blocks_per_grid_x = (rows + threads_per_block_2d[0] - 1) // threads_per_block_2d[0]
blocks_per_grid_y = (cols + threads_per_block_2d[1] - 1) // threads_per_block_2d[1]
blocks_per_grid_2d = (blocks_per_grid_x, blocks_per_grid_y)


add_arrays_2d[blocks_per_grid_2d, threads_per_block_2d](d_a_2d, d_b_2d, d_c_2d)


c_2d = d_c_2d.copy_to_host()


print("First 5x5 sub-array of 2D result:")
print(c_2d[:5, :5])

First 5x5 sub-array of 2D result:
[[  0.   2.   4.   6.   8.]
 [ 32.  34.  36.  38.  40.]
 [ 64.  66.  68.  70.  72.]
 [ 96.  98. 100. 102. 104.]
 [128. 130. 132. 134. 136.]]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
